In [17]:
# utils/admin.py

import ctypes
import sys
from typing import NoReturn


class Admin:
    """Utilities for checking and requesting administrator privileges."""

    @staticmethod
    def is_admin() -> bool:
        """Return True if the current process is running as administrator."""
        try:
            return bool(ctypes.windll.shell32.IsUserAnAdmin())
        except Exception:
            return False

    @staticmethod
    def elevate() -> NoReturn:
        """Restart the current Python program with administrator privileges.

        Displays the Windows UAC prompt. If the user approves,
        the current process exits and an elevated copy is started.

        Raises:
            RuntimeError: If elevation could not be started.
        """
        result = ctypes.windll.shell32.ShellExecuteW(
            None,
            "runas",
            sys.executable,
            " ".join(f'"{arg}"' for arg in sys.argv),
            None,
            1,
        )

        if result <= 32:
            raise RuntimeError("Failed to request administrator privileges.")

        sys.exit()

    @classmethod
    def require_admin(cls) -> None:
        """Ensure the current process is running with administrator privileges.

        If the process is not elevated, Windows will display the UAC
        prompt and restart the application as administrator.
        """
        if not cls.is_admin():
            cls.elevate()

In [18]:
# * Network Tool for Agent

import socket
import subprocess
from typing import Any, Dict, List, Optional
# from src.utils.admin import Admin

import psutil

from base_tool import BaseTool


class NetworkTool(BaseTool):
    """Tool for managing and inspecting Windows network state.

    Provides methods to control Wi-Fi, inspect local and public IP
    addresses, test connectivity, resolve DNS, and enumerate network
    adapters.
    """

    def __init__(self) -> None:
        """Initialize the network tool."""
        pass

    @property
    def name(self) -> str:
        return "network"

    @property
    def description(self) -> str:
        return "Manage and inspect Windows network operations."

    def execute(self, action: str, **kwargs: Any) -> Any:
        actions = {
            "wifi_on": self.wifi_on,
            "wifi_off": self.wifi_off,
            "ip": self.ip,
            "public_ip": self.public_ip,
            "ping": self.ping,
            "dns_lookup": self.dns_lookup,
            "internet_available": self.internet_available,
            "adapters": self.adapters,
            "connect_wifi": self.connect_wifi,
            "disconnect_wifi": self.disconnect_wifi,
        }

        if action not in actions:
            raise ValueError(f"Unknown action: {action}")

        return actions[action](**kwargs)

    # ==========================================================
    # Wi-Fi Radio Control
    # ==========================================================

    def wifi_on(self, interface: str = "Wi-Fi") -> str:
        """Enable the Wi-Fi network adapter.

        Args:
            interface: Name of the wireless network interface.

        Returns:
            Status message.
        """
        Admin.require_admin()

        subprocess.run(
            ["netsh", "interface", "set", "interface", interface, "enabled"],
            check=True,
        )

        return f"Wi-Fi interface '{interface}' enabled."

    def wifi_off(self, interface: str = "Wi-Fi") -> str:
        """Disable the Wi-Fi network adapter.

        Args:
            interface: Name of the wireless network interface.

        Returns:
            Status message.
        """
        Admin.require_admin()
        
        subprocess.run(
            ["netsh", "interface", "set", "interface", interface, "disabled"],
            check=True,
        )

        return f"Wi-Fi interface '{interface}' disabled."

    # ==========================================================
    # IP Information
    # ==========================================================

    def ip(self) -> Dict[str, str]:
        """Get the local machine's hostname and private IP address.

        Returns:
            Dictionary containing:
                hostname: The machine's network hostname.
                local_ip: The primary local IP address.
        """
        hostname = socket.gethostname()

        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
            sock.connect(("8.8.8.8", 80))
            local_ip = sock.getsockname()[0]
            sock.close()
        except OSError:
            local_ip = socket.gethostbyname(hostname)

        return {
            "hostname": hostname,
            "local_ip": local_ip,
        }

    def public_ip(self, timeout: float = 5.0) -> str:
        """Get the machine's public-facing IP address.

        Args:
            timeout: Request timeout in seconds.

        Returns:
            The public IP address as a string.

        Raises:
            RuntimeError: If the public IP could not be determined.
        """
        import urllib.request

        try:
            with urllib.request.urlopen(
                "https://api.ipify.org", timeout=timeout
            ) as response:
                return response.read().decode("utf-8").strip()
        except Exception as exc:
            raise RuntimeError(f"Could not determine public IP: {exc}") from exc

    # ==========================================================
    # Connectivity
    # ==========================================================

    def ping(self, host: str = "8.8.8.8", count: int = 4) -> Dict[str, Any]:
        """Ping a host to test reachability and latency.

        Args:
            host: Hostname or IP address to ping.
            count: Number of echo requests to send.

        Returns:
            Dictionary containing:
                host: The target host.
                success: Whether the host responded.
                output: Raw command output.
        """
        result = subprocess.run(
            ["ping", "-n", str(count), host],
            capture_output=True,
            text=True,
        )

        return {
            "host": host,
            "success": result.returncode == 0,
            "output": result.stdout,
        }

    def dns_lookup(self, hostname: str) -> str:
        """Resolve a hostname to an IP address.

        Args:
            hostname: The hostname to resolve.

        Returns:
            The resolved IP address.

        Raises:
            RuntimeError: If the hostname could not be resolved.
        """
        try:
            return socket.gethostbyname(hostname)
        except socket.gaierror as exc:
            raise RuntimeError(f"DNS lookup failed for '{hostname}': {exc}") from exc

    def internet_available(self, timeout: float = 3.0) -> bool:
        """Check whether an internet connection is currently available.

        Args:
            timeout: Connection timeout in seconds.

        Returns:
            True if internet is reachable, False otherwise.
        """
        try:
            sock = socket.create_connection(("8.8.8.8", 53), timeout=timeout)
            sock.close()
            return True
        except OSError:
            return False

    # ==========================================================
    # Adapters
    # ==========================================================

    def adapters(self) -> Dict[str, List[Dict[str, Any]]]:
        """Enumerate network adapters and their addresses/status.

        Returns:
            Dictionary mapping adapter name to a list of address
            entries (family, address, netmask, is_up).
        """
        addrs = psutil.net_if_addrs()
        stats = psutil.net_if_stats()

        result: Dict[str, List[Dict[str, Any]]] = {}

        for name, entries in addrs.items():
            is_up = stats[name].isup if name in stats else False
            result[name] = [
                {
                    "family": str(entry.family),
                    "address": entry.address,
                    "netmask": entry.netmask,
                    "is_up": is_up,
                }
                for entry in entries
            ]

        return result

    # ==========================================================
    # Wi-Fi Network Connections
    # ==========================================================

    def connect_wifi(self, ssid: str, profile: Optional[str] = None) -> str:
        """Connect to a Wi-Fi network using a saved profile.

        Args:
            ssid: Name of the Wi-Fi network to connect to.
            profile: Profile name to use, if different from the SSID.

        Returns:
            Status message.
        """
        profile_name = profile or ssid

        subprocess.run(
            ["netsh", "wlan", "connect", f"name={profile_name}", f"ssid={ssid}"],
            check=True,
        )

        return f"Connecting to Wi-Fi network '{ssid}'."

    def disconnect_wifi(self, interface: str = "Wi-Fi") -> str:
        """Disconnect from the current Wi-Fi network.

        Args:
            interface: Name of the wireless network interface.

        Returns:
            Status message.
        """
        subprocess.run(
            ["netsh", "wlan", "disconnect", f"interface={interface}"],
            check=True,
        )

        return "Wi-Fi disconnected."

In [19]:
n=NetworkTool()

In [20]:
n.ip()

{'hostname': 'LENOVO-LOQ-15IRX9', 'local_ip': '10.147.244.243'}

In [22]:
n.execute("wifi_off")

SystemExit: 

In [24]:
n.wifi_off()
%tb

SystemExit: 

d:\Windows_Agent\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
